<a href="https://colab.research.google.com/github/NahomKidane/ai301-labs/blob/main/m04-classification/guided-labs/AI301_Guided_Lab_Part1_Library_A_Task_Specific_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 Guided Lab: Classification, Part 1 · A Task Specific Model

**Module 4 · Classification**

> **File > Save a copy in Drive** before you start. The runtime can stay on CPU. This notebook
> downloads one model of roughly 500 MB and one dataset of roughly 1 MB.

Module 3 ended with a vector for a token and a vector for a whole sentence. This notebook uses a
model that already turns a sentence into a label, runs it over a thousand movie reviews, and
measures how often it is right.

The module has two routes to a classifier:

- A model already trained for the task, used as it is. This part.
- A frozen embedding model with a classifier you train yourself. Part 2.

In [ ]:
# Colab already has torch, numpy, tqdm, scikit-learn and matplotlib, so only these two are missing.
!pip install -q transformers datasets

---

## Part 1 · The data

Every measurement in this notebook is against one dataset, so the first thing to know is what is in
it and how it is split.

- **Python note:** `load_dataset` returns a dictionary of splits. `data["train"]` is one split, and
  indexing a split with a number returns one row as a dictionary of its columns.

> **Predict before running.**
>
> Rotten Tomatoes holds 5,331 positive and 5,331 negative reviews.
>
> 1. The dataset ships in three splits. How many rows would you put in each?
> 2. What do you expect one row to contain?

**Your prediction:**

_Type here._

> Dataset card: [cornell-movie-review-data/rotten_tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes).
> Read it for the split sizes and for what the label integers mean, since both decide how you read
> every number later in this notebook.

In [ ]:
from datasets import load_dataset

data = load_dataset("rotten_tomatoes")

print(data)
print()
print("one training row:", data["train"][0])

**Interpret the output.**

1. Record the three split sizes. Which is the largest, and which will you measure on?
2. The label is an integer. From the row you printed, which integer means positive?
3. Those splits were fixed by whoever published the dataset, not by you. Why does that matter when
   you compare your number against a number someone else reports?

**Your answer:**

_Type here._

---

## Part 2 · A model already trained for this task

Some models on the Hub are published with a classification head already trained. You give one a
sentence and it gives back a label. Nothing is trained here.

The model used below was trained on tweets, not on movie reviews. That is on purpose. It tells you
what a model does when the text it meets is not the text it was trained on.

`pipeline` wraps the tokenizer and the model into one callable:

```python
pipe = pipeline(model=model_path, tokenizer=model_path, top_k=None, device=device)
```

| Argument | What it does |
|---|---|
| `model` | the repository to load the weights and the classification head from |
| `tokenizer` | the tokenizer to use, which must be the one the model was trained with |
| `top_k=None` | return every label with its score, rather than only the winner |
| `device` | `0` for the first GPU, `-1` for the CPU |

Calling it runs four steps in order: tokenize the text, run the model, turn the output numbers into
probabilities, and return one score per label.

- **Python note:** `torch.cuda.is_available()` returns `True` only when a GPU is attached to the
  runtime. Writing the device this way means the same cell runs either way, instead of failing on a
  CPU runtime.

> **Predict before running.**
>
> 1. How many labels do you expect back for one review?
> 2. The dataset has two classes. Does that mean the model has two?

**Your prediction:**

_Type here._

> Model card: [cardiffnlp/twitter-roberta-base-sentiment-latest](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest).
> Read it for the label names and the training data, since the label names decide how you read the
> output of the next cell.

In [ ]:
# quiet the loading report and the progress bar warning, neither is an error
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from transformers import pipeline
import torch

model_path = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# use the GPU if the runtime has one, otherwise the CPU
device = 0 if torch.cuda.is_available() else -1
print("running on", "GPU" if device == 0 else "CPU")

pipe = pipeline(
    model=model_path,
    tokenizer=model_path,
    top_k=None,
    device=device,
)

print(pipe("A real letdown from a director who should know better."))

**Interpret the output.**

1. How many labels came back, and what are they called?
2. The scores for one review add up to something. What?
3. Nothing in this dataset is labelled neutral. What are you going to do with that label?

**Your answer:**

_Type here._

---

## Part 3 · Running it over the whole test set

One review tells you the shape of the output. A thousand tell you whether the model is any good.

- **Python note:** `KeyDataset(split, "text")` hands the pipeline one column at a time instead of
  loading the whole split into a list. `tqdm` wraps it to draw a progress bar.

The model returns three scores and the dataset has two classes, so the loop below compares the
negative score against the positive score and ignores neutral. Reading the scores by label name
rather than by position matters: the order labels come back in is a property of the model, not
something to rely on.

> **Predict before running.**
>
> A review comes back with neutral 0.70, negative 0.20, positive 0.10.
>
> 1. Which label does the loop below give it?
> 2. Is that the right call, and what would you do instead?

**Your prediction:**

_Type here._

In [ ]:
import numpy as np
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

y_pred = []

for output in tqdm(pipe(KeyDataset(data["test"], "text")), total=len(data["test"])):
    # output is a list of dictionaries, one per label
    scores = {}
    for item in output:
        scores[item["label"].lower()] = item["score"]

    negative_score = scores["negative"]
    positive_score = scores["positive"]

    # 0 is negative and 1 is positive, matching the dataset labels
    if positive_score > negative_score:
        y_pred.append(1)
    else:
        y_pred.append(0)

print("predictions made:", len(y_pred))
print("first ten:", y_pred[:10])
print("first ten true labels:", data["test"]["label"][:10])

**Interpret the output.**

1. How many of the first ten match the true labels?
2. Count how many of your predictions are 1. Given that the test split is balanced, what would you
   expect that count to be?
3. Neutral was dropped. Name one kind of review where that will cost you.

**Your answer:**

_Type here._

---

## Part 4 · The confusion matrix

A single accuracy number hides which mistakes were made. The confusion matrix does not. It counts
the four things that can happen when a prediction meets a true label.

| | predicted negative | predicted positive |
|---|---|---|
| **truly negative** | true negative | false positive |
| **truly positive** | false negative | true positive |

Everything else in this part is arithmetic on those four counts.

- **Python note:** `classification_report` prints precision, recall and F1 for each class and three
  summary rows underneath. `ConfusionMatrixDisplay` draws the table above as a heat map.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


def evaluate(y_true, y_pred, class_names=("negative", "positive")):
    """Print the per class scores and draw the confusion matrix."""
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

    matrix = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(matrix, display_labels=class_names)
    display.plot(cmap="Blues", colorbar=False)
    plt.title("Task specific model on Rotten Tomatoes")
    plt.show()


evaluate(data["test"]["label"], y_pred)

**Interpret the output.**

1. Read the four cells of the matrix out loud as sentences. "The model said positive about N reviews
   that were actually negative."
2. Which mistake does this model make more often, calling a negative review positive or the reverse?
3. Look at the per class rows. Is the model equally good at both classes?

**Your answer:**

_Type here._

---

## Part 5 · Why accuracy is not the number to quote

Accuracy is the fraction of predictions that are right. On this dataset it is a fair summary,
because the two classes are the same size. That is unusual.

The cell below is not about movie reviews. It is the smallest example of why accuracy stops meaning
anything when the classes are not balanced.

> **Predict before running.**
>
> A million documents. A hundred of them are about pie. A classifier answers "not pie" every single
> time, without reading anything.
>
> 1. What accuracy does it score?
> 2. How many pie documents does it find?

**Your prediction:**

_Type here._

In [ ]:
total = 1000000
pie = 100
not_pie = total - pie

# the classifier always answers "not pie", so it is right on every document that is not pie
correct = not_pie

print("accuracy:", correct / total)
print("pie documents found:", 0, "out of", pie)

**Interpret the output.**

1. Write the accuracy down. Would you ship this classifier?
2. Which of the four cells of the confusion matrix is large here, and which is zero?

Precision and recall are what you quote instead. Both are about one class at a time.

**Precision** is: of everything the model called positive, how much really was. It is the number to
watch when a false alarm is expensive.

**Recall** is: of everything that really was positive, how much the model found. It is the number to
watch when a miss is expensive.

**F1** is the harmonic mean of the two, which is a single number that stays low unless both are
high.

Now look at the three summary rows `classification_report` printed in Part 4.

- `accuracy` is one number over everything.
- `macro avg` averages the per class scores, giving each class equal weight.
- `weighted avg` averages them weighted by how many examples each class has.

On a balanced test set those two rows are close to identical. On the pie problem they would not be.
Quote the macro average when you want each class to count the same, and say which one you quoted.

**Your answer:**

_Type here._

> **Your turn.**
>
> The loop in Part 3 ran on the test split. Change it to run on `data["validation"]` instead, and
> evaluate that. Do the numbers move much? What does the size of the change tell you about how much
> to trust a single number from a single split?

In [ ]:
# Your turn: run the model on the validation split and evaluate it.

---

## Check yourself

- Say in one sentence what `top_k=None` changes about what the pipeline returns.
- The model has three labels and the dataset has two. Say where that gap was handled and how.
- Give one example where high precision matters more than high recall, and one the other way round.
- Say why the macro average and the weighted average are nearly the same on this dataset.

## Summary

A task specific model carries a classification head that was already trained, so you load it and
call it.

The pipeline runs tokenizer and model together and returns one score per label.

The labels a model returns belong to the model, not to your dataset. Reading them by name, and
deciding what to do with the ones your dataset does not have, is part of the work.

The confusion matrix holds the four counts. Precision, recall and F1 are arithmetic on those counts.

Accuracy is only a fair summary when the classes are balanced, and most real problems are not.

A number is not comparable to someone else's number unless the dataset, the split and the averaging
method are the same.

## References

- Alammar and Grootendorst, *Hands-On Large Language Models*, Chapter 4, Text Classification.
  This notebook is adapted from the book's own Chapter 4 notebook.
- Book code repository: [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models)
- The book's Chapter 4 notebook:
  [on GitHub](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)
  and [open in Colab](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter04/Chapter%204%20-%20Text%20Classification.ipynb)
- Jurafsky and Martin, *Speech and Language Processing*, Chapter 4, Logistic Regression and Text
  Classification, section on precision, recall and F1.
- Model card: [cardiffnlp/twitter-roberta-base-sentiment-latest](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)
- Dataset card: [cornell-movie-review-data/rotten_tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes)
- [transformers pipeline reference](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [sklearn classification_report reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)